# Find Variable Sites from RSV F Protein Alignment

This notebook identifies variable (polymorphic) sites in RSV F protein sequences by comparing them to the RSV_Long_F reference sequence.

In [ ]:
# Parameters - will be overridden by papermill
strain = "RSV-A"
alignment_file = "data/RSV_F_seqs/RSV-A_with_ref.fasta"
output_file = "results/polymorphisms/RSV-A_variable_sites.csv"
min_differences = 2

In [ ]:
import pandas as pd
from Bio import SeqIO
from collections import Counter
import os

## Configuration

In [ ]:
print(f"Analyzing strain: {strain}")
print(f"Input alignment: {alignment_file}")
print(f"Output file: {output_file}")
print(f"Minimum differences threshold: {min_differences}")

## Define function to analyze variable sites

In [ ]:
def analyze_variable_sites_detailed(fasta_file, min_differences=2):
    """
    Analyze variable sites in a multiple sequence alignment.
    
    Parameters:
    -----------
    fasta_file : str
        Path to FASTA alignment file with reference as first sequence
    min_differences : int
        Minimum number of sequences with a mutation for it to be reported
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: site, wildtype, mutant, mutation_count, mutation_type
    """
    # Parse all sequences
    sequences = list(SeqIO.parse(fasta_file, 'fasta'))
    
    # First sequence is the reference
    reference = sequences[0]
    ref_name = reference.id
    ref_seq = str(reference.seq)
    
    print(f"Reference sequence: {ref_name}")
    print(f"Reference length: {len(ref_seq)} amino acids")
    print(f"Total sequences in alignment: {len(sequences)}")
    
    # Track mutations at each position
    variable_sites = []
    
    # Iterate through each position in the reference
    for pos in range(len(ref_seq)):
        site = pos + 1  # 1-based numbering
        ref_aa = ref_seq[pos]
        
        # Skip stop codons
        if ref_aa == '*':
            continue
        
        # Collect all amino acids at this position across sequences
        mutations = []
        for seq in sequences[1:]:  # Skip reference
            if pos < len(seq.seq):
                seq_aa = str(seq.seq[pos])
                # Only count if different from reference and not a stop or gap
                if seq_aa != ref_aa and seq_aa != '*' and seq_aa != '-':
                    mutations.append(seq_aa)
        
        # Count mutation frequencies
        if mutations:
            mutation_counts = Counter(mutations)
            
            # Report mutations meeting the threshold
            for mutant_aa, count in mutation_counts.items():
                if count >= min_differences:
                    variable_sites.append({
                        'site': site,
                        'wildtype': ref_aa,
                        'mutant': mutant_aa,
                        'mutation_count': count,
                        'mutation_type': f"{ref_aa}→{mutant_aa}"
                    })
    
    # Create DataFrame and sort
    df = pd.DataFrame(variable_sites)
    if len(df) > 0:
        df = df.sort_values(['site', 'mutation_count'], ascending=[True, False])
    
    print(f"\nFound {len(df)} polymorphisms at {df['site'].nunique()} sites")
    print(f"(appearing in at least {min_differences} sequences)")
    
    return df

## Analyze alignment

In [ ]:
# Run analysis
variable_sites = analyze_variable_sites_detailed(
    alignment_file, 
    min_differences=min_differences
)

# Display first rows
print(f"\nFirst 20 polymorphisms for {strain}:")
display(variable_sites.head(20))

## Summary statistics

In [ ]:
print(f"Total number of polymorphisms: {len(variable_sites)}")
print(f"Number of variable sites: {variable_sites['site'].nunique()}")
print(f"\nMost variable sites:")
top_sites = variable_sites.groupby('site').size().sort_values(ascending=False).head(10)
for site, count in top_sites.items():
    wt = variable_sites[variable_sites['site'] == site]['wildtype'].iloc[0]
    print(f"  Site {site} ({wt}): {count} different mutations observed")

print(f"\nMost common polymorphisms:")
top_mutations = variable_sites.nlargest(10, 'mutation_count')[['site', 'mutation_type', 'mutation_count']]
display(top_mutations)

## Save results

In [ ]:
# Create output directory if needed
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Save to CSV
variable_sites.to_csv(output_file, index=False)
print(f"\nSaved results to: {output_file}")